In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

In [3]:
new_data=pd.read_csv(r"C:\Users\HEMANATH\Desktop\customer-churn-prediction\data\processed\Cleaned_data_for_eda.csv")
new_data

,Unnamed: 0,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,...,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value
0,0,33.964131,-118.272783,Male,No,No,No,2,Yes,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,1,34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,2,34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,3,34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1
4,4,34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,...,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7027,7038,34.341737,-116.539416,Female,No,No,No,72,Yes,No,...,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Bank transfer (automatic),21.15,1419.40,0
7028,7039,34.667815,-117.536183,Male,No,Yes,Yes,24,Yes,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50,0
7029,7040,34.559882,-115.637164,Female,No,Yes,Yes,72,Yes,Yes,...,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90,0
7030,7041,34.167800,-116.864330,Female,No,Yes,Yes,11,No,No phone service,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45,0


In [4]:
new_data.drop("Unnamed: 0",axis=1,inplace=True)

In [5]:
X = new_data.drop("Churn Value", axis=1)

In [6]:
y = new_data["Churn Value"]
y

0       1
1       1
2       1
3       1
4       1
       ..
7027    0
7028    0
7029    0
7030    0
7031    0
Name: Churn Value, Length: 7032, dtype: int64

In [7]:
cat_cols=X.select_dtypes(include="object").columns
num_cols=X.select_dtypes(exclude="object").columns

print(cat_cols)
print(num_cols)

Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method'],
      dtype='object')
Index(['Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges',
       'Total Charges'],
      dtype='object')


In [8]:
num_transformer=StandardScaler()

In [9]:
cat_transformer=OneHotEncoder(
    handle_unknown="ignore"
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            num_transformer,
            num_cols
        ),
        (
            "cat",
            cat_transformer,
            cat_cols
        )
    ]
)

In [13]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.2,shuffle=True,stratify=y)

In [14]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

lr_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
predictions=lr_pipeline.predict(X_test)
predictions

array([0, 1, 0, ..., 0, 0, 0], shape=(1407,))

In [16]:
probs=lr_pipeline.predict_proba(X_test)[:,1]
probs

array([0.03390735, 0.61378625, 0.00588499, ..., 0.15138594, 0.03102103,
       0.00122699], shape=(1407,))

In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        eval_metric='logloss'
    ))
])

xgb_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
from sklearn.tree import DecisionTreeClassifier

dt_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

dt_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
from sklearn.metrics import accuracy_score

models = {
    "Logistic Regression": lr_pipeline,
    "Random Forest": rf_pipeline,
    "Decision Tree": dt_pipeline,
    "XGBoost": xgb_pipeline
}

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)

    print(f"{name}: {acc:.4f}")

Logistic Regression: 0.8081
Random Forest: 0.7896
Decision Tree: 0.7783
XGBoost: 0.7818


In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    prob = model.predict_proba(X_test)[:,1]

    results.append([
        name,
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, prob)
    ])

In [22]:
results_df=pd.DataFrame(results,columns=["Model","Accuracy","Precision","Recall","F1","ROC_AUC"])

print(results_df.sort_values("ROC_AUC",ascending=False))

                 Model  Accuracy  Precision    Recall        F1   ROC_AUC
0  Logistic Regression  0.808102   0.649425  0.604278  0.626039  0.842686
3              XGBoost  0.781805   0.603077  0.524064  0.560801  0.839047
2        Decision Tree  0.778252   0.568889  0.684492  0.621359  0.835140
1        Random Forest  0.789623   0.628289  0.510695  0.563422  0.832217
